In [ ]:
# In[1]:

import pandas as pd
import pytz

# Reuse variables where possible
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to convert min/max timestamp to UTC+8 timezone-aware ISO strings
def ts_min_max_iso(s):
    if s.empty:
        return (None, None)
    s_ts = pd.to_datetime(s.dropna().astype(int), unit='s', utc=True)
    s_ts = s_ts.dt.tz_convert(tz)
    return (s_ts.min().isoformat(), s_ts.max().isoformat())

# Helper to get top N lexicographic unique values as a comma-joined string (compact)
def top_lex(series, n=20):
    if series is None or series.empty:
        return ""
    vals = sorted(pd.Series(series.dropna().astype(str).unique()))
    top = vals[:n]
    return ", ".join(top)

# Summarize function for each dataframe
def summarize_df(df, name_col=None, message_col=None):
    row_count = int(df.shape[0])
    min_ts, max_ts = ts_min_max_iso(df['timestamp']) if 'timestamp' in df.columns else (None, None)
    top_cmdb = top_lex(df['cmdb_id']) if 'cmdb_id' in df.columns else ""
    if name_col:
        top_names = top_lex(df[name_col])
    elif message_col:
        top_names = top_lex(df[message_col])
    else:
        top_names = ""
    first5 = df.head(5)
    return {
        "row_count": row_count,
        "min_timestamp": min_ts,
        "max_timestamp": max_ts,
        "top_cmdb_id_samples": top_cmdb,
        "top_name_samples": top_names,
        "first_5_rows": first5  # keep as DataFrame for compact display
    }

# Create summaries
metric_summary = summarize_df(metric_df, name_col='kpi_name')
trace_summary = summarize_df(trace_df, name_col='trace_name')
log_summary = summarize_df(log_df, name_col='log_name')
error_summary = summarize_df(error_df, message_col='message')

# Build a compact pandas DataFrame summary (stringified top lists) for quick overview
summary_df = pd.DataFrame([
    {
        "file": "metric.csv",
        "row_count": metric_summary["row_count"],
        "min_timestamp": metric_summary["min_timestamp"],
        "max_timestamp": metric_summary["max_timestamp"],
        "top_cmdb_id_samples": metric_summary["top_cmdb_id_samples"],
        "top_kpi_name_samples": metric_summary["top_name_samples"]
    },
    {
        "file": "trace.csv",
        "row_count": trace_summary["row_count"],
        "min_timestamp": trace_summary["min_timestamp"],
        "max_timestamp": trace_summary["max_timestamp"],
        "top_cmdb_id_samples": trace_summary["top_cmdb_id_samples"],
        "top_trace_name_samples": trace_summary["top_name_samples"]
    },
    {
        "file": "log.csv",
        "row_count": log_summary["row_count"],
        "min_timestamp": log_summary["min_timestamp"],
        "max_timestamp": log_summary["max_timestamp"],
        "top_cmdb_id_samples": log_summary["top_cmdb_id_samples"],
        "top_log_name_samples": log_summary["top_name_samples"]
    },
    {
        "file": "error_logs.csv",
        "row_count": error_summary["row_count"],
        "min_timestamp": error_summary["min_timestamp"],
        "max_timestamp": error_summary["max_timestamp"],
        "top_cmdb_id_samples": error_summary["top_cmdb_id_samples"],
        "top_message_samples": error_summary["top_name_samples"]
    }
])

# Prepare first-5 DataFrames for display (compact)
first5_metric = metric_summary["first_5_rows"]
first5_trace = trace_summary["first_5_rows"]
first5_log = log_summary["first_5_rows"]
first5_error = error_summary["first_5_rows"]

# Final displayed variables (compact JSON-like overview + first 5 rows for each file)
summary_df, first5_metric, first5_trace, first5_log, first5_error

```
Out[1]:
```
```python
# Summarize previous execution results in plain English (compact)
rows = []
for idx, r in summary_df.iterrows():
    file = r['file']
    rc = int(r['row_count'])
    mn = r['min_timestamp'] if pd.notna(r['min_timestamp']) else "None"
    mx = r['max_timestamp'] if pd.notna(r['max_timestamp']) else "None"
    top_cmdb = r['top_cmdb_id_samples'] if pd.notna(r['top_cmdb_id_samples']) else ""
    # Select the appropriate top-name column if present
    if file == 'metric.csv':
        top_names = r.get('top_kpi_name_samples', "") or ""
    elif file == 'trace.csv':
        top_names = r.get('top_trace_name_samples', "") or ""
    elif file == 'log.csv':
        top_names = r.get('top_log_name_samples', "") or ""
    else:
        top_names = r.get('top_message_samples', "") or ""
    # Truncate long sample lists for compactness
    def trunc(s, n=180):
        s = str(s)
        return s if len(s) <= n else s[:n].rstrip() + " ..."
    rows.append(f"{file}: rows={rc}; min_ts={mn}; max_ts={mx}; top_cmdb_samples=[{trunc(top_cmdb)}]; top_name_samples=[{trunc(top_names)}]")

summary_text = " | ".join(rows)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count              min_timestamp              max_timestamp                                top_cmdb_id_samples                               top_kpi_name_samples                             top_trace_name_samples            top_log_name_samples top_message_samples
0      metric.csv     325376  2022-03-21T13:00:00+08:00  2022-03-21T14:00:00+08:00  adservice, adservice-0, adservice-1, adservice...  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...                                                NaN                             NaN                 NaN
1       trace.csv      36240  2022-03-21T13:30:00+08:00  2022-03-21T13:59:00+08:00  adservice-0, adservice-1, adservice-2, adservi...                                                NaN  trace.from_checkoutservice-0.duration_mean, tr...                             NaN                 NaN
2         log.csv       1842  2022-03-21T13:30:00+08:00  2022-03-21T14:00:00+08:00  adservice-0, adservice-1, adservice-2, cartser...                                                NaN                                                NaN  log.error_count, log.row_count                 NaN
3  error_logs.csv          0                       None                       None                                                                                                   NaN                                                NaN                             NaN                    ,     timestamp    cmdb_id        kpi_name      value
0  1647838800  adservice  app.grpc.count  215.00000
1  1647838800  adservice    app.grpc.mrt    0.00274
2  1647838800  adservice     app.grpc.rr  100.00000
3  1647838800  adservice     app.grpc.sr  100.00000
4  1647838800  adservice  app.http.count   24.00000,     timestamp      cmdb_id                           trace_name      value
0  1647840600  adservice-0  trace.from_frontend-0.duration_mean   0.000018
1  1647840600  adservice-0   trace.from_frontend-0.duration_p95   0.000028
2  1647840600  adservice-0     trace.from_frontend-0.error_rate   0.000000
3  1647840600  adservice-0      trace.from_frontend-0.row_count  18.000000
4  1647840600  adservice-0  trace.from_frontend-1.duration_mean   0.000020,     timestamp      cmdb_id         log_name  value
0  1647840600  adservice-0  log.error_count    0.0
1  1647840600  adservice-0    log.row_count  124.0
2  1647840600  adservice-1  log.error_count    0.0
3  1647840600  adservice-1    log.row_count  108.0
4  1647840600  adservice-2  log.error_count    0.0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse metric_df if already loaded; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timezone object for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Create timezone-aware timestamp column (timestamps are epoch seconds UTC -> convert then to Asia/Shanghai)
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and n_total per (cmdb_id, kpi_name)
group_keys = ['cmdb_id', 'kpi_name']
p95_series = metric_df.groupby(group_keys)['value'].quantile(0.95).reset_index(name='P95')
n_total_series = metric_df.groupby(group_keys).size().reset_index(name='n_total')
groups_df = pd.merge(p95_series, n_total_series, on=group_keys)

# Merge P95 back into full series for later comparisons
metric_with_p95 = pd.merge(metric_df, p95_series, on=group_keys, how='left')

# 2) Filter to incident window in UTC+8
start = pd.Timestamp('2022-03-21 13:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 14:00:00', tz=tz)
window_mask = (metric_with_p95['ts'] >= start) & (metric_with_p95['ts'] <= end)
window_df = metric_with_p95[window_mask].copy()

# 3) For each pair compute window stats
# n_window and max_window_value
window_agg = window_df.groupby(group_keys).agg(
    n_window=('value', 'size'),
    max_window_value=('value', 'max')
).reset_index()

# n_anom: in-window points where value >= P95
window_df['is_anom'] = window_df['value'] >= window_df['P95']
n_anom_series = window_df.groupby(group_keys)['is_anom'].sum().reset_index(name='n_anom')

# earliest_anom_ts per group (tz-aware). Keep as ISO string in UTC+8
anom_df = window_df[window_df['is_anom']]
earliest_anom = anom_df.groupby(group_keys)['ts'].min().reset_index(name='earliest_anom_ts')

# Merge window stats into groups_df
groups_df = groups_df.merge(window_agg, on=group_keys, how='left')
groups_df = groups_df.merge(n_anom_series, on=group_keys, how='left')
groups_df = groups_df.merge(earliest_anom, on=group_keys, how='left')

# Fill NaNs for groups with no window data
groups_df['n_window'] = groups_df['n_window'].fillna(0).astype(int)
groups_df['n_anom'] = groups_df['n_anom'].fillna(0).astype(int)
groups_df['max_window_value'] = groups_df['max_window_value']  # keep NaN if no window
groups_df['earliest_anom_ts'] = groups_df['earliest_anom_ts']  # NaT if none

# severity = max_window_value - P95 if there are window points; else 0
groups_df['severity'] = groups_df.apply(
    lambda r: 0.0 if r['n_window'] == 0 or pd.isna(r['max_window_value']) else float(r['max_window_value'] - r['P95']),
    axis=1
)

# Format earliest_anom_ts to ISO strings (compact) or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return x.isoformat()

groups_df['earliest_anom_ts'] = groups_df['earliest_anom_ts'].apply(fmt_ts)

# Select required columns
out_df = groups_df[['cmdb_id', 'kpi_name', 'P95', 'n_total', 'n_window', 'n_anom', 'earliest_anom_ts', 'max_window_value', 'severity']].copy()

# Sorting: prioritize rows with n_anom > 0, then by severity desc. Then take top 20.
out_df['has_anom'] = out_df['n_anom'] > 0
out_df = out_df.sort_values(by=['has_anom', 'severity'], ascending=[False, False])

# Limit to top 20
result_df = out_df.head(20).drop(columns=['has_anom']).reset_index(drop=True)

# Keep numeric columns compact (round P95, max_window_value, severity)
result_df['P95'] = result_df['P95'].round(6)
result_df['max_window_value'] = result_df['max_window_value'].round(6)
result_df['severity'] = result_df['severity'].round(6)

# Display the compact result (top up to 20 rows)
result_df

```
Out[2]:
```
```python
# Summarize the previously computed top 20 results in plain English (compact)
# Reuse result_df from prior computation
top = result_df.copy()

# Build a concise plain-English summary using the top 5 entries for detail
n_returned = top.shape[0]
lines = []
lines.append(f"Returned {n_returned} (cmdb_id, kpi_name) pairs sorted by severity (desc).")
if n_returned == 0:
    summary = "\n".join(lines)
else:
    lines.append("Top entries (showing up to 5):")
    for i, row in top.head(5).iterrows():
        cmdb = row['cmdb_id']
        kpi = row['kpi_name']
        p95 = f"{row['P95']:.6e}"
        n_total = int(row['n_total'])
        n_window = int(row['n_window'])
        n_anom = int(row['n_anom'])
        earliest = row['earliest_anom_ts'] if row['earliest_anom_ts'] is not None else "None"
        maxv = f"{row['max_window_value']:.6e}" if not pd.isna(row['max_window_value']) else "None"
        sev = f"{row['severity']:.6e}"
        lines.append(f"{i+1}. {cmdb} | {kpi} — P95={p95}; n_total={n_total}; n_window={n_window}; n_anom={n_anom}; earliest_anom_ts={earliest}; max_window_value={maxv}; severity={sev}")
    # Aggregate observation lines
    lines.append("")
    lines.append("Observations:")
    lines.append("- Highest severities are for adservice2 and adservice process-CPU related metrics and for several currencyservice pods' container memory RSS.")
    lines.append("- Several adservice2 metrics (GC and memory) show all window points flagged as anomalies (n_anom = n_window = 31), indicating sustained spikes across the incident window.")
    lines.append("- Currencyservice pods (currencyservice-0/1/2 and currencyservice2-0) show very large memory RSS increases during the window (severity on the order of 1.6e+08), implying memory pressure on those containers.")
    lines.append("- Timestamps are in UTC+8; earliest anomaly times in the top rows occur between 13:32 and 13:57 (UTC+8) within the 2022-03-21 13:30–14:00 window.")
    lines.append("")
    lines.append("Conclusion: The most severe anomalies during the incident window are concentrated on process CPU metrics for adservice/adservice2 and on container memory RSS for currencyservice pods. These are the top candidate (cmdb_id, kpi_name) pairs for further investigation.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           P95  n_total  n_window  n_anom           earliest_anom_ts  max_window_value      severity
0           adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  1.148933e+11       61        31       4  2022-03-21T13:57:00+08:00      1.156808e+11  7.875000e+08
1            adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  2.226773e+12       61        31       4  2022-03-21T13:57:00+08:00      2.227358e+12  5.850000e+08
2    currencyservice-1              container.node-6.container_memory_rss  3.503718e+07       61        31       5  2022-03-21T13:36:00+08:00      1.996022e+08  1.645650e+08
3    currencyservice-2              container.node-6.container_memory_rss  3.392307e+07       61        31      10  2022-03-21T13:32:00+08:00      1.981563e+08  1.642332e+08
4    currencyservice-0              container.node-6.container_memory_rss  3.491021e+07       61        31       3  2022-03-21T13:39:00+08:00      1.912566e+08  1.563464e+08
5            adservice  runtime.java_lang_Threading_CurrentThreadUserTime  2.134508e+11       61        31       4  2022-03-21T13:51:00+08:00      2.135500e+11  9.916667e+07
6           adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  8.414767e+09       61        31       4  2022-03-21T13:57:00+08:00      8.505232e+09  9.046508e+07
7           adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  7.823333e+09       61        31       4  2022-03-21T13:57:00+08:00      7.905000e+09  8.166667e+07
8            adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  2.338539e+11       61        31       4  2022-03-21T13:56:00+08:00      2.339279e+11  7.402660e+07
9   currencyservice2-0              container.node-6.container_memory_rss  3.553690e+07       61        31       3  2022-03-21T13:39:00+08:00      8.937882e+07  5.384192e+07
10          adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  5.677259e+09       61        31       4  2022-03-21T13:57:00+08:00      5.722254e+09  4.499498e+07
11           adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  1.691900e+11       61        31       4  2022-03-21T13:57:00+08:00      1.692333e+11  4.336489e+07
12          adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.667000e+04       61        31      31  2022-03-21T13:30:00+08:00      2.337176e+07  2.334509e+07
13          adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.664300e+04       61        31      31  2022-03-21T13:30:00+08:00      2.337164e+07  2.334500e+07
14          adservice2  runtime.java_lang_Memory_HeapMemoryUsage_commi...  3.928474e+07       61        31      31  2022-03-21T13:30:00+08:00      5.607424e+07  1.678950e+07
15          adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  7.274496e+06       61        31      31  2022-03-21T13:30:00+08:00      2.287206e+07  1.559757e+07
16          adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  7.274496e+06       61        31      31  2022-03-21T13:30:00+08:00      2.287206e+07  1.559757e+07
17          adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  6.923968e+06       61        31      31  2022-03-21T13:30:00+08:00      2.209261e+07  1.516864e+07
18          adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  6.923968e+06       61        31      31  2022-03-21T13:30:00+08:00      2.209261e+07  1.516864e+07
19          adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.702950e+07       61        31      31  2022-03-21T13:30:00+08:00      3.864166e+07  1.161216e+07```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse trace_df and tz if available, otherwise load/define
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column is integer and create timezone-aware ts in UTC+8
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Group keys
group_keys = ['cmdb_id', 'trace_name']

# 1) Compute global P95 and n_total per (cmdb_id, trace_name)
p95_df = trace_df.groupby(group_keys)['value'].quantile(0.95).reset_index(name='P95')
n_total_df = trace_df.groupby(group_keys).size().reset_index(name='n_total')
groups_df = pd.merge(p95_df, n_total_df, on=group_keys)

# Merge P95 into full series for comparisons
trace_with_p95 = pd.merge(trace_df, p95_df, on=group_keys, how='left')

# 2) Filter to incident window (inclusive) in UTC+8
start = pd.Timestamp('2022-03-21 13:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 14:00:00', tz=tz)
window_mask = (trace_with_p95['ts'] >= start) & (trace_with_p95['ts'] <= end)
window_df = trace_with_p95[window_mask].copy()

# 3) Compute window aggregates
window_agg = window_df.groupby(group_keys).agg(
    n_window=('value', 'size'),
    max_window_value=('value', 'max')
).reset_index()

# Determine anomalies in window: value >= P95
window_df['is_anom'] = window_df['value'] >= window_df['P95']
n_anom_df = window_df.groupby(group_keys)['is_anom'].sum().reset_index(name='n_anom')

# Earliest anomaly timestamp per group
anom_df = window_df[window_df['is_anom']]
earliest_anom = anom_df.groupby(group_keys)['ts'].min().reset_index(name='earliest_anom_ts')

# Merge window stats into groups_df
groups_df = groups_df.merge(window_agg, on=group_keys, how='left')
groups_df = groups_df.merge(n_anom_df, on=group_keys, how='left')
groups_df = groups_df.merge(earliest_anom, on=group_keys, how='left')

# Fill NaNs for groups with no window data
groups_df['n_window'] = groups_df['n_window'].fillna(0).astype(int)
groups_df['n_anom'] = groups_df['n_anom'].fillna(0).astype(int)
# max_window_value may remain NaN if no window points
# earliest_anom_ts may be NaT; keep for formatting

# 3 continued) severity calculation
def compute_severity(row):
    if row['n_window'] == 0 or pd.isna(row['max_window_value']):
        return 0.0
    return float(row['max_window_value'] - row['P95'])

groups_df['severity'] = groups_df.apply(compute_severity, axis=1)

# Format earliest_anom_ts to ISO string or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return x.isoformat()

groups_df['earliest_anom_ts'] = groups_df['earliest_anom_ts'].apply(fmt_ts)

# Select required columns
out_df = groups_df[['cmdb_id', 'trace_name', 'P95', 'n_total', 'n_window', 'n_anom', 'earliest_anom_ts', 'max_window_value', 'severity']].copy()

# Sorting: prioritize n_anom>0 first, then by severity desc
out_df['has_anom'] = out_df['n_anom'] > 0
out_df = out_df.sort_values(by=['has_anom', 'severity'], ascending=[False, False])

# Keep top 20 rows
result_trace = out_df.head(20).drop(columns=['has_anom']).reset_index(drop=True)

# Round numeric columns for compact display
result_trace['P95'] = result_trace['P95'].round(6)
result_trace['max_window_value'] = result_trace['max_window_value'].round(6)
result_trace['severity'] = result_trace['severity'].round(6)

# Display the compact result (up to 20 rows)
result_trace

```
Out[3]:
```
```python
# Summarize the trace analysis results in plain English (compact)
try:
    result_trace
except NameError:
    import pandas as pd
    result_trace = pd.DataFrame()  # fallback, should not happen in normal flow

n_returned = result_trace.shape[0]
lines = []
lines.append(f"Returned {n_returned} (cmdb_id, trace_name) pairs ranked by severity (n_anom>0 prioritized).")
if n_returned == 0:
    lines.append("No trace pairs available in the result.")
else:
    lines.append("Top entries (up to 5 shown):")
    for i, row in result_trace.head(5).iterrows():
        cmdb = row['cmdb_id']
        trace = row['trace_name']
        p95 = f"{row['P95']:.6f}"
        n_total = int(row['n_total'])
        n_window = int(row['n_window'])
        n_anom = int(row['n_anom'])
        earliest = row['earliest_anom_ts'] if row['earliest_anom_ts'] is not None else "None"
        maxv = f"{row['max_window_value']:.6f}" if not pd.isna(row['max_window_value']) else "None"
        sev = f"{row['severity']:.6f}"
        lines.append(f"{i+1}. {cmdb} | {trace} — P95={p95}; n_total={n_total}; n_window={n_window}; n_anom={n_anom}; earliest_anom_ts={earliest}; max={maxv}; severity={sev}")
    lines.append("")
    lines.append("Key observations:")
    lines.append("- The highest severities are dominated by frontend-related row_count traces (trace.self.row_count) and traces between frontend and productcatalogservice, indicating elevated request counts during the incident window.")
    lines.append("- Many affected pairs have full window coverage (n_window=30) with several anomalous points (n_anom>0), meaning sustained elevated traffic within 2022-03-21 13:30–14:00 (UTC+8).")
    lines.append("- A notable latency-related spike appears for cartservice/frontend traces (duration_p95), where max window latency far exceeds the global P95, indicating latency incidents as well.")
    lines.append("- Earliest anomaly timestamps in top rows occur between 13:30 and 13:57 (UTC+8).")
    lines.append("")
    lines.append("Conclusion: The trace data points to increased request volumes centered on frontend pods and their calls to productcatalogservice (and a notable latency spike involving cartservice). These trace pairs are prime candidates for further investigation.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name          P95  n_total  n_window  n_anom           earliest_anom_ts  max_window_value   severity
0                 frontend-2                         trace.self.row_count  1010.150000       30        30       2  2022-03-21T13:30:00+08:00         1069.0000  58.850000
1                 frontend-0                         trace.self.row_count   870.000000       30        30       3  2022-03-21T13:49:00+08:00          917.0000  47.000000
2                frontend2-0                         trace.self.row_count  1418.400000       30        30       2  2022-03-21T13:38:00+08:00         1462.0000  43.600000
3                 frontend-1                         trace.self.row_count  1028.100000       30        30       2  2022-03-21T13:37:00+08:00         1050.0000  21.900000
4                 frontend-0   trace.to_productcatalogservice-1.row_count   132.550000       30        30       2  2022-03-21T13:49:00+08:00          153.0000  20.450000
5                 frontend-0   trace.to_productcatalogservice-2.row_count   132.550000       30        30       2  2022-03-21T13:57:00+08:00          153.0000  20.450000
6    productcatalogservice-1              trace.from_frontend-0.row_count   132.550000       30        30       2  2022-03-21T13:49:00+08:00          153.0000  20.450000
7    productcatalogservice-2              trace.from_frontend-0.row_count   132.550000       30        30       2  2022-03-21T13:57:00+08:00          153.0000  20.450000
8                 frontend-0   trace.to_productcatalogservice-0.row_count   133.000000       30        30       4  2022-03-21T13:49:00+08:00          152.0000  19.000000
9    productcatalogservice-0              trace.from_frontend-0.row_count   133.000000       30        30       4  2022-03-21T13:49:00+08:00          152.0000  19.000000
10                frontend-1   trace.to_productcatalogservice-2.row_count   154.750000       30        30       2  2022-03-21T13:44:00+08:00          173.0000  18.250000
11   productcatalogservice-2              trace.from_frontend-1.row_count   154.750000       30        30       2  2022-03-21T13:44:00+08:00          173.0000  18.250000
12                frontend-1   trace.to_productcatalogservice-0.row_count   154.200000       30        30       2  2022-03-21T13:44:00+08:00          172.0000  17.800000
13   productcatalogservice-0              trace.from_frontend-1.row_count   154.200000       30        30       2  2022-03-21T13:44:00+08:00          172.0000  17.800000
14                frontend-1   trace.to_productcatalogservice-1.row_count   155.200000       30        30       2  2022-03-21T13:44:00+08:00          172.0000  16.800000
15   productcatalogservice-1              trace.from_frontend-1.row_count   155.200000       30        30       2  2022-03-21T13:44:00+08:00          172.0000  16.800000
16               frontend2-0  trace.to_productcatalogservice2-0.row_count   665.750000       30        30       2  2022-03-21T13:38:00+08:00          681.0000  15.250000
17  productcatalogservice2-0             trace.from_frontend2-0.row_count   665.750000       30        30       2  2022-03-21T13:38:00+08:00          681.0000  15.250000
18             cartservice-1           trace.from_frontend-0.duration_p95     0.002045       30        30       2  2022-03-21T13:52:00+08:00           12.5868  12.584755
19                frontend-0          trace.to_cartservice-1.duration_p95     0.002045       30        30       2  2022-03-21T13:52:00+08:00           12.5868  12.584755```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing variables where possible
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp -> timezone-aware ts in UTC+8
log_df['ts'] = pd.to_datetime(log_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Group keys
group_keys = ['cmdb_id', 'log_name']

# 1) Global P95 and n_total per (cmdb_id, log_name)
p95_df = log_df.groupby(group_keys)['value'].quantile(0.95).reset_index(name='P95')
n_total_df = log_df.groupby(group_keys).size().reset_index(name='n_total')
groups_df = pd.merge(p95_df, n_total_df, on=group_keys)

# Merge P95 into full series
log_with_p95 = pd.merge(log_df, p95_df, on=group_keys, how='left')

# 2) Filter to incident window (inclusive) in UTC+8
start = pd.Timestamp('2022-03-21 13:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 14:00:00', tz=tz)
window_mask = (log_with_p95['ts'] >= start) & (log_with_p95['ts'] <= end)
window_df = log_with_p95[window_mask].copy()

# 3) Window aggregates
window_agg = window_df.groupby(group_keys).agg(
    n_window=('value', 'size'),
    max_window_value=('value', 'max')
).reset_index()

# Anomalies: value >= P95
window_df['is_anom'] = window_df['value'] >= window_df['P95']
n_anom_df = window_df.groupby(group_keys)['is_anom'].sum().reset_index(name='n_anom')

# Earliest anomaly timestamp per group
anom_df = window_df[window_df['is_anom']]
earliest_anom = anom_df.groupby(group_keys)['ts'].min().reset_index(name='earliest_anom_ts')

# Merge window stats into groups_df
groups_df = groups_df.merge(window_agg, on=group_keys, how='left')
groups_df = groups_df.merge(n_anom_df, on=group_keys, how='left')
groups_df = groups_df.merge(earliest_anom, on=group_keys, how='left')

# Fill NaNs for groups with no window data
groups_df['n_window'] = groups_df['n_window'].fillna(0).astype(int)
groups_df['n_anom'] = groups_df['n_anom'].fillna(0).astype(int)

# severity calculation
def compute_severity(row):
    if row['n_window'] == 0 or pd.isna(row['max_window_value']):
        return 0.0
    return float(row['max_window_value'] - row['P95'])

groups_df['severity'] = groups_df.apply(compute_severity, axis=1)

# Format earliest_anom_ts to ISO or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return x.isoformat()

groups_df['earliest_anom_ts'] = groups_df['earliest_anom_ts'].apply(fmt_ts)

# Select required columns
out_df = groups_df[['cmdb_id', 'log_name', 'P95', 'n_total', 'n_window', 'n_anom', 'earliest_anom_ts', 'max_window_value', 'severity']].copy()

# Sorting: prioritize n_anom>0 first, then by severity desc
out_df['has_anom'] = out_df['n_anom'] > 0
out_df = out_df.sort_values(by=['has_anom', 'severity'], ascending=[False, False])

# Top 20 rows
result_log = out_df.head(20).drop(columns=['has_anom']).reset_index(drop=True)

# Round numeric columns for compact display
result_log['P95'] = result_log['P95'].round(6)
result_log['max_window_value'] = result_log['max_window_value'].round(6)
result_log['severity'] = result_log['severity'].round(6)

# Display compact result (up to 20 rows)
result_log

```
Out[4]:
```
```python
# Summarize the log analysis results in plain English (compact)
try:
    df = result_log
except NameError:
    df = None

if df is None or df.empty:
    summary = "No log analysis results available."
else:
    n = df.shape[0]
    lines = []
    lines.append(f"Returned {n} (cmdb_id, log_name) pairs ranked by severity (n_anom>0 prioritized).")
    lines.append("Top entries (up to 5 shown):")
    for i, row in df.head(5).iterrows():
        cmdb = row['cmdb_id']
        logname = row['log_name']
        p95 = f"{row['P95']:.2f}"
        n_total = int(row['n_total'])
        n_window = int(row['n_window'])
        n_anom = int(row['n_anom'])
        earliest = row['earliest_anom_ts'] if row['earliest_anom_ts'] is not None else "None"
        maxv = f"{row['max_window_value']:.1f}" if not pd.isna(row['max_window_value']) else "None"
        sev = f"{row['severity']:.2f}"
        lines.append(f"{i+1}. {cmdb} | {logname} — P95={p95}; n_total={n_total}; n_window={n_window}; n_anom={n_anom}; earliest_anom_ts={earliest}; max={maxv}; severity={sev}")
    lines.append("")
    lines.append("Key observations:")
    lines.append("- The largest severities are on log.row_count for cartservice and frontend pods (e.g., cartservice-0, frontend-2, cartservice-2), indicating elevated request counts during the incident window.")
    lines.append("- Many affected pairs have full window coverage (n_window = 30 or 31) with some anomalous points (n_anom > 0), meaning sustained higher log counts across 2022-03-21 13:30–14:00 (UTC+8).")
    lines.append("- Other services with notable increases include currencyservice and productcatalogservice.")
    lines.append("")
    lines.append("Conclusion: Log data points to increased request/log activity concentrated on frontend and cartservice pods (and also currency/productcatalog services). These (cmdb_id, log_name) pairs are priority candidates for further investigation.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id       log_name      P95  n_total  n_window  n_anom           earliest_anom_ts  max_window_value  severity
0             cartservice-0  log.row_count   885.85       30        30       2  2022-03-21T13:44:00+08:00            1003.0    117.15
1                frontend-2  log.row_count  1548.50       31        31       2  2022-03-21T13:30:00+08:00            1650.0    101.50
2             cartservice-2  log.row_count   893.50       31        31       2  2022-03-21T13:52:00+08:00             992.0     98.50
3                frontend-1  log.row_count  1552.00       31        31       2  2022-03-21T13:37:00+08:00            1606.0     54.00
4         currencyservice-1  log.row_count   797.50       31        31       2  2022-03-21T13:30:00+08:00             846.0     48.50
5             cartservice-1  log.row_count   880.00       31        31       3  2022-03-21T13:44:00+08:00             928.0     48.00
6   productcatalogservice-1  log.row_count  1029.50       30        30       2  2022-03-21T13:44:00+08:00            1065.0     35.50
7         currencyservice-0  log.row_count   804.00       31        31       2  2022-03-21T13:34:00+08:00             837.0     33.00
8         currencyservice-2  log.row_count   800.50       31        31       2  2022-03-21T13:54:00+08:00             828.0     27.50
9   productcatalogservice-0  log.row_count  1040.10       30        30       2  2022-03-21T13:44:00+08:00            1065.0     24.90
10  productcatalogservice-2  log.row_count  1030.70       30        30       2  2022-03-21T13:44:00+08:00            1051.0     20.30
11        checkoutservice-0  log.row_count    84.55       30        30       2  2022-03-21T13:34:00+08:00              96.0     11.45
12        shippingservice-1  log.row_count   110.55       30        30       2  2022-03-21T13:54:00+08:00             122.0     11.45
13               frontend-0  log.row_count  1335.00       31        31       2  2022-03-21T13:49:00+08:00            1346.0     11.00
14        checkoutservice-1  log.row_count    72.40       30        30       2  2022-03-21T13:34:00+08:00              82.0      9.60
15  recommendationservice-2  log.row_count   214.50       31        31       2  2022-03-21T13:44:00+08:00             224.0      9.50
16  recommendationservice-1  log.row_count   213.00       31        31       2  2022-03-21T13:44:00+08:00             221.0      8.00
17        checkoutservice-2  log.row_count    77.05       30        30       2  2022-03-21T13:30:00+08:00              85.0      7.95
18        shippingservice-0  log.row_count   112.10       30        30       2  2022-03-21T13:44:00+08:00             119.0      6.90
19              adservice-2  log.row_count   114.20       30        30       2  2022-03-21T13:55:00+08:00             120.0      5.80```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing metric_df and tz if present; otherwise load/define
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure ts column (timezone-aware UTC+8) exists
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware dtype
    if metric_df['ts'].dtype == 'int64' or metric_df['ts'].dtype == 'float64' or metric_df['ts'].dtype == 'object':
        metric_df['ts'] = pd.to_datetime(metric_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Define the target pairs (as list of tuples)
target_pairs = [
    ('adservice2', 'runtime.java_lang_OperatingSystem_ProcessCpuTime'),
    ('adservice', 'runtime.java_lang_OperatingSystem_ProcessCpuTime'),
    ('currencyservice-1', 'container.node-6.container_memory_rss'),
    ('currencyservice-2', 'container.node-6.container_memory_rss'),
    ('currencyservice-0', 'container.node-6.container_memory_rss'),
    ('adservice', 'runtime.java_lang_Threading_CurrentThreadUserTime'),
    ('adservice2', 'runtime.java_lang_Threading_CurrentThreadCpuTime'),
    ('adservice2', 'runtime.java_lang_Threading_CurrentThreadUserTime'),
    ('adservice', 'runtime.java_lang_Threading_CurrentThreadCpuTime'),
    ('currencyservice2-0', 'container.node-6.container_memory_rss')
]

# Prepare incident window
start = pd.Timestamp('2022-03-21 13:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 14:00:00', tz=tz)

# Filter metric_df to only target pairs for efficiency
mask_pairs = metric_df.apply(lambda r: (r['cmdb_id'], r['kpi_name']) in set(target_pairs), axis=1)
df_targets = metric_df[mask_pairs].copy()

# Compute global P95 per pair using all rows (rule: compute BEFORE filtering by time)
p95_df = df_targets.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index(name='P95')

# Ensure all target pairs present in p95_df; if missing, add rows with NaN P95
pairs_df = pd.DataFrame(target_pairs, columns=['cmdb_id', 'kpi_name'])
p95_df = pairs_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Container for runs
runs = []

# Process each pair individually
for _, row in p95_df.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    P95 = row['P95']
    # Subset full series for this pair (already used above to compute P95)
    pair_full = df_targets[(df_targets['cmdb_id'] == cmdb) & (df_targets['kpi_name'] == kpi)].copy()
    # n_total (full series)
    n_total = int(pair_full.shape[0])
    # Filter to incident window
    pair_window = pair_full[(pair_full['ts'] >= start) & (pair_full['ts'] <= end)].copy().sort_values('ts')
    n_window = int(pair_window.shape[0])
    if n_window == 0:
        # No window points -> skip runs (severity 0 by rule). Still continue to next pair.
        continue
    # Compute is_anom using global P95 (if P95 is NaN, treat no anomalies)
    if pd.isna(P95):
        pair_window['is_anom'] = False
    else:
        pair_window['is_anom'] = pair_window['value'] >= P95
    n_anom = int(pair_window['is_anom'].sum())
    max_window_value = pair_window['value'].max()
    # Exceedance ratio guard
    exceedance_ratio = None
    if (not pd.isna(P95)) and P95 != 0:
        exceedance_ratio = (float(max_window_value) - float(P95)) / float(P95)
    else:
        exceedance_ratio = None

    # Identify consecutive anomaly runs where is_anom == True
    anom_points = pair_window[pair_window['is_anom']].copy().sort_values('ts')
    if anom_points.empty:
        continue

    # Compute minute gaps between consecutive anomaly points
    anom_points['prev_ts'] = anom_points['ts'].shift(1)
    anom_points['gap_min'] = (anom_points['ts'] - anom_points['prev_ts']) / pd.Timedelta(minutes=1)
    # Start a new run when gap != 1
    anom_points['run_group'] = (anom_points['gap_min'] != 1).cumsum()
    # Aggregate runs
    runs_agg = anom_points.groupby('run_group').agg(
        run_start_ts=('ts', 'min'),
        run_end_ts=('ts', 'max'),
        run_length_minutes=('ts', lambda s: int((s.max() - s.min()) / pd.Timedelta(minutes=1)) + 1),
        run_count=('ts', 'size')
    ).reset_index(drop=True)

    # Only keep runs with length >= 2 minutes (rule)
    runs_agg = runs_agg[runs_agg['run_length_minutes'] >= 2].copy()
    if runs_agg.empty:
        continue

    # For each run, produce output row with pair-level n_window, n_anom, max_window_value, exceedance_ratio
    for _, rrun in runs_agg.iterrows():
        runs.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'P95': float(P95) if not pd.isna(P95) else None,
            'run_start_ts': rrun['run_start_ts'].isoformat(),
            'run_end_ts': rrun['run_end_ts'].isoformat(),
            'run_length_minutes': int(rrun['run_length_minutes']),
            'n_window': n_window,
            'n_anom': n_anom,
            'max_window_value': float(max_window_value),
            'exceedance_ratio': float(exceedance_ratio) if exceedance_ratio is not None else None
        })

# Build result DataFrame, sort by run_start_ts ascending, limit to first 20 runs
if runs:
    runs_df = pd.DataFrame(runs)
    # parse run_start_ts to sort reliably
    runs_df['run_start_ts_dt'] = pd.to_datetime(runs_df['run_start_ts']).dt.tz_convert(tz)
    runs_df = runs_df.sort_values('run_start_ts_dt').drop(columns=['run_start_ts_dt']).reset_index(drop=True)
    # Round numeric columns compactly
    runs_df['P95'] = runs_df['P95'].map(lambda x: round(x,6) if pd.notna(x) else None)
    runs_df['max_window_value'] = runs_df['max_window_value'].map(lambda x: round(x,6))
    runs_df['exceedance_ratio'] = runs_df['exceedance_ratio'].map(lambda x: round(x,6) if pd.notna(x) else None)
    result_runs = runs_df.head(20)[['cmdb_id','kpi_name','P95','run_start_ts','run_end_ts','run_length_minutes','n_window','n_anom','max_window_value','exceedance_ratio']].copy()
else:
    # Empty result
    result_runs = pd.DataFrame(columns=['cmdb_id','kpi_name','P95','run_start_ts','run_end_ts','run_length_minutes','n_window','n_anom','max_window_value','exceedance_ratio'])

# Display compact result (up to 20 rows)
result_runs

```
Out[5]:
```
```python
# Summarize the run detection results from result_runs (already computed)
try:
    result_runs
except NameError:
    result_runs = None

if result_runs is None or result_runs.empty:
    summary = "No consecutive anomaly runs (length>=2 minutes) were detected for the requested pairs in the incident window."
else:
    lines = []
    lines.append(f"Detected {len(result_runs)} consecutive anomaly runs (length>=2 minutes) for the requested (cmdb_id, kpi_name) pairs:")
    for i, r in result_runs.reset_index(drop=True).iterrows():
        idx = i + 1
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        p95 = "{:.6e}".format(r['P95']) if pd.notna(r['P95']) else "None"
        start = r['run_start_ts']
        end = r['run_end_ts']
        length = int(r['run_length_minutes'])
        n_window = int(r['n_window'])
        n_anom = int(r['n_anom'])
        maxv = "{:.6e}".format(r['max_window_value'])
        ratio = "{:.6f}".format(r['exceedance_ratio']) if pd.notna(r['exceedance_ratio']) else "None"
        lines.append(f"{idx}. {cmdb} | {kpi} — P95={p95}; run={start}→{end} ({length} min); n_window={n_window}; n_anom={n_anom}; max={maxv}; exceedance_ratio={ratio}")
    lines.append("")
    # Observations
    lines.append("Observations:")
    lines.append("- Currencyservice pods (currencyservice-2, -1, -0) show long memory-RSS anomaly runs (3–10 minutes) with very large exceedance ratios (~4.48–4.84), indicating strong memory pressure during the 13:30–14:00 window.")
    lines.append("- currencyservice2-0 also shows a 3-minute memory-RSS run with moderate exceedance (~1.52x).")
    lines.append("- adservice2 and adservice show short process-CPU anomaly runs starting at ~13:57 and lasting 3–4 minutes; their exceedance ratios are small (~0.00026–0.0108), indicating modest CPU spikes relative to their historical P95.")
    lines.append("")
    lines.append("Conclusion:")
    lines.append("- Primary concern: memory pressure on currencyservice pods during the incident window; investigate memory usage, GC behavior, or possible leaks/OOM causes.")
    lines.append("- Secondary: brief CPU increases on adservice/adservice2 around 13:57—monitor but less severe.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           P95               run_start_ts                 run_end_ts  run_length_minutes  n_window  n_anom  max_window_value  exceedance_ratio
0   currencyservice-2              container.node-6.container_memory_rss  3.392307e+07  2022-03-21T13:32:00+08:00  2022-03-21T13:41:00+08:00                  10        31      10      1.981563e+08          4.841343
1   currencyservice-1              container.node-6.container_memory_rss  3.503718e+07  2022-03-21T13:36:00+08:00  2022-03-21T13:40:00+08:00                   5        31       5      1.996022e+08          4.696867
2   currencyservice-0              container.node-6.container_memory_rss  3.491021e+07  2022-03-21T13:39:00+08:00  2022-03-21T13:41:00+08:00                   3        31       3      1.912566e+08          4.478529
3  currencyservice2-0              container.node-6.container_memory_rss  3.553690e+07  2022-03-21T13:39:00+08:00  2022-03-21T13:41:00+08:00                   3        31       3      8.937882e+07          1.515099
4          adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  1.148933e+11  2022-03-21T13:57:00+08:00  2022-03-21T14:00:00+08:00                   4        31       4      1.156808e+11          0.006854
5           adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  2.226773e+12  2022-03-21T13:57:00+08:00  2022-03-21T14:00:00+08:00                   4        31       4      2.227358e+12          0.000263
6          adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  8.414767e+09  2022-03-21T13:57:00+08:00  2022-03-21T14:00:00+08:00                   4        31       4      8.505232e+09          0.010751
7          adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  7.823333e+09  2022-03-21T13:57:00+08:00  2022-03-21T14:00:00+08:00                   4        31       4      7.905000e+09          0.010439
8           adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  2.338539e+11  2022-03-21T13:58:00+08:00  2022-03-21T14:00:00+08:00                   3        31       4      2.339279e+11          0.000317```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing trace_df and tz if present; otherwise load/define
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware ts column exists (UTC+8)
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure proper dtype
    if not pd.api.types.is_datetime64tz_dtype(trace_df['ts']):
        trace_df['ts'] = pd.to_datetime(trace_df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Select pairs where cmdb_id contains 'currencyservice'
mask_cs = trace_df['cmdb_id'].astype(str).str.contains('currencyservice', na=False)
cs_df = trace_df[mask_cs].copy()

# If none found, return empty DataFrame with required columns
cols = ['cmdb_id','trace_name','P95','n_total','n_window','n_anom','earliest_anom_ts','max_window_value','severity']
if cs_df.empty:
    result_cs = pd.DataFrame(columns=cols)
else:
    # 1) Global P95 and n_total per (cmdb_id, trace_name) using all available rows
    group_keys = ['cmdb_id','trace_name']
    p95_all = cs_df.groupby(group_keys)['value'].quantile(0.95).reset_index(name='P95')
    n_total = cs_df.groupby(group_keys).size().reset_index(name='n_total')
    groups = pd.merge(p95_all, n_total, on=group_keys, how='outer')

    # 2) Filter to incident window (inclusive)
    start = pd.Timestamp('2022-03-21 13:30:00', tz=tz)
    end = pd.Timestamp('2022-03-21 14:00:00', tz=tz)
    window_df = cs_df[(cs_df['ts'] >= start) & (cs_df['ts'] <= end)].copy()

    # 3) Compute window aggregates per pair
    window_agg = window_df.groupby(group_keys).agg(
        n_window=('value','size'),
        max_window_value=('value','max')
    ).reset_index()

    # anomalies in window where value >= global P95 (merge P95 into window rows)
    window_with_p95 = pd.merge(window_df, p95_all, on=group_keys, how='left')
    window_with_p95['is_anom'] = False
    mask_p95_valid = window_with_p95['P95'].notna()
    window_with_p95.loc[mask_p95_valid, 'is_anom'] = window_with_p95.loc[mask_p95_valid, 'value'] >= window_with_p95.loc[mask_p95_valid, 'P95']
    n_anom = window_with_p95.groupby(group_keys)['is_anom'].sum().reset_index(name='n_anom')

    # earliest anomaly timestamp per group
    anom_points = window_with_p95[window_with_p95['is_anom']].copy()
    if not anom_points.empty:
        earliest_anom = anom_points.groupby(group_keys)['ts'].min().reset_index(name='earliest_anom_ts')
    else:
        earliest_anom = pd.DataFrame(columns=group_keys+['earliest_anom_ts'])

    # Merge all stats together
    summary = groups.merge(window_agg, on=group_keys, how='left') \
                    .merge(n_anom, on=group_keys, how='left') \
                    .merge(earliest_anom, on=group_keys, how='left')

    # Fill NaNs for counts
    summary['n_window'] = summary['n_window'].fillna(0).astype(int)
    summary['n_anom'] = summary['n_anom'].fillna(0).astype(int)
    # max_window_value may be NaN if no window points
    # earliest_anom_ts may be NaT; format to ISO or None
    summary['earliest_anom_ts'] = summary['earliest_anom_ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)

    # severity = max_window_value - P95; if no window points set severity = 0
    def compute_sev(r):
        if r['n_window'] == 0 or pd.isna(r['max_window_value']) or pd.isna(r['P95']):
            return 0.0
        return float(r['max_window_value'] - r['P95'])
    summary['severity'] = summary.apply(compute_sev, axis=1)

    # Select required columns and round numeric values compactly
    result_cs = summary[['cmdb_id','trace_name','P95','n_total','n_window','n_anom','earliest_anom_ts','max_window_value','severity']].copy()
    result_cs['P95'] = result_cs['P95'].round(6)
    result_cs['max_window_value'] = result_cs['max_window_value'].round(6)
    result_cs['severity'] = result_cs['severity'].round(6)

    # 4) Sort: prioritize n_anom>0 first, then severity desc
    result_cs['has_anom'] = result_cs['n_anom'] > 0
    result_cs = result_cs.sort_values(by=['has_anom','severity'], ascending=[False, False]).drop(columns=['has_anom']).reset_index(drop=True)

    # Limit to top 20 rows
    result_cs = result_cs.head(20)

# Final compact output variable
result_cs

```
Out[6]:
```
Returned 20 (cmdb_id, trace_name) pairs for components containing "currencyservice", prioritized by presence of anomalies and severity.

Top findings (highest severity first):
1. currencyservice-1 | trace.from_frontend-1.row_count — P95=110.10; n_total=30; n_window=30; n_anom=2; earliest_anom_ts=2022-03-21T13:37:00+08:00; max=118.0; severity=7.9  
2. currencyservice2-0 | trace.from_frontend2-0.row_count — P95=436.20; n_total=30; n_window=30; n_anom=2; earliest_anom_ts=2022-03-21T13:38:00+08:00; max=444.0; severity=7.8  
3. currencyservice-2 | trace.from_frontend-1.row_count — P95=109.55; n_total=30; n_window=30; n_anom=2; earliest_anom_ts=2022-03-21T13:37:00+08:00; max=117.0; severity=7.45  
4. currencyservice-0 | trace.from_frontend-1.row_count — P95=110.00; n_total=30; n_window=30; n_anom=3; earliest_anom_ts=2022-03-21T13:37:00+08:00; max=117.0; severity=7.0

Additional notable items (smaller severity but n_anom>0):
- Multiple currencyservice pods show elevated row_count traces from frontend (several trace.from_frontend-*.row_count entries) with n_window=30 and a few anomalous points each (n_anom typically 2–3). Earliest anomaly times cluster around 13:30–13:38 (UTC+8).
- Some checkoutservice→currencyservice traces and duration_p95 metrics have minor exceedances (small absolute severities), e.g., trace.from_checkoutservice*.row_count and duration_p95 entries.
- All listed pairs have full coverage in the incident window (n_window = 30), so anomalies represent elevated activity during the whole window rather than sparse sampling.

Conclusion / recommendation:
- Currencyservice pods are experiencing increased request volumes (front-end traffic) during 2022-03-21 13:30–14:00 (UTC+8), with the strongest signals in trace.from_frontend row_count metrics across multiple pods. Investigate upstream frontend traffic patterns, load distribution, and currencyservice capacity (threads, service latency, scaling) as the next step.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name         P95  n_total  n_window  n_anom           earliest_anom_ts  max_window_value  severity
0    currencyservice-1            trace.from_frontend-1.row_count  110.100000       30        30       2  2022-03-21T13:37:00+08:00        118.000000  7.900000
1   currencyservice2-0           trace.from_frontend2-0.row_count  436.200000       30        30       2  2022-03-21T13:38:00+08:00        444.000000  7.800000
2    currencyservice-2            trace.from_frontend-1.row_count  109.550000       30        30       2  2022-03-21T13:37:00+08:00        117.000000  7.450000
3    currencyservice-0            trace.from_frontend-1.row_count  110.000000       30        30       3  2022-03-21T13:37:00+08:00        117.000000  7.000000
4    currencyservice-0            trace.from_frontend-0.row_count   91.300000       30        30       2  2022-03-21T13:30:00+08:00         98.000000  6.700000
5    currencyservice-1            trace.from_frontend-0.row_count   91.300000       30        30       2  2022-03-21T13:30:00+08:00         98.000000  6.700000
6    currencyservice-2            trace.from_frontend-0.row_count   91.400000       30        30       2  2022-03-21T13:30:00+08:00         98.000000  6.600000
7    currencyservice-2            trace.from_frontend-2.row_count  112.400000       30        30       2  2022-03-21T13:34:00+08:00        119.000000  6.600000
8    currencyservice-0            trace.from_frontend-2.row_count  112.400000       30        30       2  2022-03-21T13:34:00+08:00        118.000000  5.600000
9    currencyservice-1            trace.from_frontend-2.row_count  112.400000       30        30       2  2022-03-21T13:34:00+08:00        118.000000  5.600000
10  currencyservice2-0    trace.from_checkoutservice2-0.row_count   21.000000       30        30       4  2022-03-21T13:38:00+08:00         24.000000  3.000000
11   currencyservice-0     trace.from_checkoutservice-2.row_count    5.000000       30        30       4  2022-03-21T13:33:00+08:00          6.000000  1.000000
12   currencyservice-2     trace.from_checkoutservice-0.row_count    5.000000       30        30       3  2022-03-21T13:34:00+08:00          6.000000  1.000000
13   currencyservice-1     trace.from_checkoutservice-2.row_count    5.550000       30        30       2  2022-03-21T13:46:00+08:00          6.000000  0.450000
14   currencyservice-2     trace.from_checkoutservice-1.row_count    4.550000       30        30       2  2022-03-21T13:33:00+08:00          5.000000  0.450000
15   currencyservice-1  trace.from_checkoutservice-0.duration_p95    0.000164       30        30       2  2022-03-21T13:34:00+08:00          0.001216  0.001052
16   currencyservice-0        trace.from_frontend-0.duration_mean    0.000131       30        30       2  2022-03-21T13:41:00+08:00          0.000742  0.000611
17   currencyservice-1  trace.from_checkoutservice-2.duration_p95    0.000172       30        30       2  2022-03-21T13:41:00+08:00          0.000712  0.000540
18   currencyservice-2  trace.from_checkoutservice-2.duration_p95    0.000155       30        30       2  2022-03-21T13:44:00+08:00          0.000656  0.000501
19  currencyservice2-0       trace.from_frontend2-0.duration_mean    0.000109       30        30       2  2022-03-21T13:41:00+08:00          0.000557  0.000448```
```